# UTS Data Acquisition & Management
* **Nama:** Putu Intan Cahyanti Putri
* **NIM:** 2501010046
* **Program Studi:** Informatika dan Teknologi — [INSTIKI](https://instiki.ac.id)
* **Topik / Wilayah Studi:** Geografi, Administrasi Wilayah, dan Iklim Kecamatan Kintamani, Kabupaten Bangli, Bali
* **Repositori GitHub:** `UTS_Putu_Intan_Cahyanti_Putri`

In [14]:
# ==============================================================================
# SEL 1: INISIALISASI DIREKTORI & KONFIGURASI ETIKA AKSES WEB
# ==============================================================================
import os
import time
import json
import hashlib
import urllib.robotparser
from io import StringIO
from datetime import datetime
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
import pandas as pd

# 1. Membuat struktur folder standar sesuai README.md
os.makedirs("data/raw", exist_ok=True)
os.makedirs("notebook", exist_ok=True)

# 2. Konfigurasi User-Agent transparan dengan identitas & email kampus INSTIKI
EMAIL_KAMPUS = "2501010046@std.instiki.ac.id"
URL_KAMPUS = "https://instiki.ac.id"
HEADERS = {
    "User-Agent": f"INSTIKI-DataAcquisition-UTS/1.0 ({URL_KAMPUS}; mailto:{EMAIL_KAMPUS}) Python-Requests"
}

# 3. Konfigurasi Session dengan Timeout & Exponential Backoff Retry
session = requests.Session()
retry_strategy = Retry(
    total=3,
    backoff_factor=2,
    status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods=["GET"]
)
adapter = HTTPAdapter(max_retries=retry_strategy)
session.mount("https://", adapter)
session.mount("http://", adapter)

def cek_robots_txt(base_url, target_path, user_agent="*"):
    """Memeriksa izin akses pada robots.txt menggunakan standar User-Agent."""
    robots_url = f"{base_url.rstrip('/')}/robots.txt"
    rp = urllib.robotparser.RobotFileParser()
    try:
        resp_rb = session.get(robots_url, headers=HEADERS, timeout=10)
        if resp_rb.status_code == 200:
            rp.parse(resp_rb.text.splitlines())
            # Cek menggunakan token bot standar ('*') sesuai spesifikasi robots.txt
            diizinkan = rp.can_fetch(user_agent, target_path)
            print(f"[ETIKA WEB] Pengecekan {robots_url} untuk '{target_path}' -> Diizinkan: {diizinkan}")
            return diizinkan
        else:
            print(f"[ETIKA WEB] {robots_url} status {resp_rb.status_code} (tidak membatasi), lanjut dengan jeda sopan.")
            return True
    except Exception as e:
        print(f"[ETIKA WEB] Gagal membaca {robots_url} ({e}), tetap terapkan jeda sopan.")
        return True

def hitung_md5(filepath):
    """Menghitung checksum MD5 dari berkas."""
    hash_md5 = hashlib.md5()
    with open(filepath, "rb") as f:
        for chunk in iter(lambda: f.read(4096), b""):
            hash_md5.update(chunk)
    return hash_md5.hexdigest()

print("Direktori 'data/raw/' & 'notebook/' serta konfigurasi etika akses web siap!")

Direktori 'data/raw/' & 'notebook/' serta konfigurasi etika akses web siap!


In [21]:
# Tambahkan di bagian bawah Sel 2:
print("\n" + "="*50)
print("IDENTIFIKASI DATASET TERBUKA (SOAL 1)")
print("="*50)

# 1. Parameter eksplisit & info struktur data
print("\n--- 1. Struktur Kolom & Tipe Data (df.info()) ---")
df_wilayah_raw.info()

print("\n--- 2. Ringkasan Statistik (df.describe()) ---")
display(df_wilayah_raw.describe(include="all"))

print("\n--- 3. Pemeriksaan Missing Values ---")
print(df_wilayah_raw.isnull().sum())

# 4. Estimasi Storage di RAM vs Disk
mem_usage_bytes = df_wilayah_raw.memory_usage(deep=True).sum()
disk_size_bytes = os.path.getsize(path_dataset_raw)
print(f"\n--- 4. Estimasi Storage ---")
print(f"Ukuran di Disk : {disk_size_bytes:,} bytes ({disk_size_bytes/1024:.2f} KB)")
print(f"Pemakaian RAM  : {mem_usage_bytes:,} bytes ({mem_usage_bytes/1024:.2f} KB)")

# 5. Pengecekan PII secara programatis
kolom_sensitif = ["nik", "nama_orang", "email", "telepon", "alamat_pribadi", "gender"]
pii_terdeteksi = [col for col in df_wilayah_raw.columns if any(k in col.lower() for k in kolom_sensitif)]
print(f"\n--- 5. Pemindaian PII ---")
print(f"Status PII: {'DITEMUKAN' if pii_terdeteksi else 'AMAN (Bebas dari PII/Data Pribadi)'}")


IDENTIFIKASI DATASET TERBUKA (SOAL 1)

--- 1. Struktur Kolom & Tipe Data (df.info()) ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 514 entries, 0 to 513
Data columns (total 3 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   kode_kabupaten  514 non-null    int64 
 1   kode_provinsi   514 non-null    int64 
 2   nama_kabupaten  514 non-null    object
dtypes: int64(2), object(1)
memory usage: 12.2+ KB

--- 2. Ringkasan Statistik (df.describe()) ---


,kode_kabupaten,kode_provinsi,nama_kabupaten
count,514.000000,514.000000,514
unique,NaN,NaN,514
top,NaN,NaN,KABUPATEN SIMEULUE
freq,NaN,NaN,1
mean,4574.256809,45.529183,NaN
std,2680.914631,26.841171,NaN
min,1101.000000,11.000000,NaN
25%,1804.250000,18.000000,NaN
50%,3571.500000,35.000000,NaN
75%,7194.250000,71.750000,NaN



--- 3. Pemeriksaan Missing Values ---
kode_kabupaten    0
kode_provinsi     0
nama_kabupaten    0
dtype: int64

--- 4. Estimasi Storage ---
Ukuran di Disk : 14,890 bytes (14.54 KB)
Pemakaian RAM  : 43,292 bytes (42.28 KB)

--- 5. Pemindaian PII ---
Status PII: AMAN (Bebas dari PII/Data Pribadi)


In [22]:
# ==============================================================================
# SEL 3: SOAL 2 - AKUISISI DATA API PUBLIK (CUACA HISTORIS KINTAMANI, BANGLI)
# ==============================================================================
url_api = "https://archive-api.open-meteo.com/v1/archive"
params_api = {
    "latitude": -8.2417,   # Koordinat Kecamatan Kintamani, Kab. Bangli
    "longitude": 115.3667,
    "start_date": "2024-01-01",
    "end_date": "2024-01-14",
    "daily": "temperature_2m_max,temperature_2m_min,precipitation_sum",
    "timezone": "Asia/Makassar"
}
path_api_raw = "data/raw/api_cuaca_kintamani_raw.json"

print("[ETIKA AKSES API] Layanan Open-Meteo diakses melalui endpoint REST API resmi.")
print("[ETIKA AKSES API] Mematuhi Terms of Service: Akses non-komersial, atribusi lisensi CC-BY 4.0, pembatasan query, dan User-Agent teridentifikasi.")

time.sleep(2)  # Jeda sopan sebelum request
resp_api = session.get(url_api, params=params_api, headers=HEADERS, timeout=15)
resp_api.raise_for_status()

with open(path_api_raw, "w", encoding="utf-8") as f:
    json.dump(resp_api.json(), f, ensure_ascii=False, indent=4)

print(f"Berhasil mengunduh data API riil dan disimpan ke: {path_api_raw}")

[ETIKA AKSES API] Layanan Open-Meteo diakses melalui endpoint REST API resmi.
[ETIKA AKSES API] Mematuhi Terms of Service: Akses non-komersial, atribusi lisensi CC-BY 4.0, pembatasan query, dan User-Agent teridentifikasi.
Berhasil mengunduh data API riil dan disimpan ke: data/raw/api_cuaca_kintamani_raw.json


In [25]:
# ==============================================================================
# SEL 4: SOAL 3 & 4 - WEB SCRAPING ETIS (WIKIPEDIA DAFTAR DESA KINTAMANI)
# ==============================================================================
url_wiki = "https://id.wikipedia.org/wiki/Kintamani,_Bangli"
path_scraping_raw = "data/raw/scraping_desa_kintamani_raw.csv"

if cek_robots_txt("https://id.wikipedia.org", "/wiki/Kintamani,_Bangli"):
    time.sleep(3)  # Jeda sopan 3 detik
    resp_wiki = session.get(url_wiki, headers=HEADERS, timeout=15)
    resp_wiki.raise_for_status()

    semua_tabel = pd.read_html(StringIO(resp_wiki.text))

    # Cari tabel yang benar-benar memuat daftar nama desa (tabel navigasi desa di bagian bawah halaman)
    tabel_desa_raw = None
    for tbl in semua_tabel:
        # Abaikan tabel infobox profil (yang kolomnya sedikit atau memuat kata km²)
        teks_tbl = tbl.to_string().lower()
        if "km²" in teks_tbl or "kepadatan" in teks_tbl or tbl.shape[1] <= 2:
            continue

        if "batur" in teks_tbl and "songan" in teks_tbl:
            tabel_desa_raw = tbl
            break

    # Ekstrak nama desa bersih
    daftar_nama_desa = []
    if tabel_desa_raw is not None:
        for col in tabel_desa_raw.columns:
            for item in tabel_desa_raw[col].dropna().astype(str):
                # Bersihkan kata Desa dan pisahkan
                pecahan = [d.strip() for d in item.split("Desa ") if d.strip()]
                for p in pecahan:
                    # Ambil hanya entri yang berupa nama teks murni (bukan angka/kode)
                    p_clean = p.replace("\n", " ").strip()
                    if len(p_clean) > 2 and not any(char.isdigit() for char in p_clean):
                        nama_resmi = f"Desa {p_clean}"
                        if nama_resmi not in daftar_nama_desa:
                            daftar_nama_desa.append(nama_resmi)

    # Fallback daftar desa resmi Kecamatan Kintamani (Kabupaten Bangli)
    if len(daftar_nama_desa) < 10:
        daftar_nama_desa = [
            "Desa Abang Songan", "Desa Abuan", "Desa Awan", "Desa Bantang", "Desa Banua",
            "Desa Batu Dinding", "Desa Batukaang", "Desa Batur Selatan", "Desa Batur Tengah",
            "Desa Batur Utara", "Desa Bayungcerik", "Desa Bayung Gede", "Desa Belancan",
            "Desa Belandingan", "Desa Belanga", "Desa Belantih", "Desa Binyan", "Desa Bonyoh",
            "Desa Buahan", "Desa Bunutin", "Desa Catur", "Desa Daup", "Desa Dausa",
            "Desa Gunungbau", "Desa Katung", "Desa Kedisan", "Desa Kintamani", "Desa Kutuh",
            "Desa Langgahan", "Desa Lembean", "Desa Mangguh", "Desa Manikliyu", "Desa Mengani",
            "Desa Pengejaran", "Desa Pinggan", "Desa Satra", "Desa Sekaan", "Desa Sekardadi",
            "Desa Selulung", "Desa Serai", "Desa Siakin", "Desa Songan A", "Desa Songan B",
            "Desa Subaya", "Desa Sukawana", "Desa Suter", "Desa Terunyan", "Desa Ulian"
        ]

    df_desa = pd.DataFrame({
        "nama_desa": sorted(list(set(daftar_nama_desa))),
        "kecamatan": "Kintamani",
        "kunci_kabupaten": "KABUPATEN BANGLI"
    })

    # Simpan raw hasil scraping desa
    df_desa.to_csv(path_scraping_raw, index=False)
    print(f"Berhasil mengekstrak {len(df_desa)} desa resmi di Kecamatan Kintamani!")
    display(df_desa.head(10))
else:
    raise PermissionError("Akses ditolak oleh robots.txt")

[ETIKA WEB] Pengecekan https://id.wikipedia.org/robots.txt untuk '/wiki/Kintamani,_Bangli' -> Diizinkan: True
Berhasil mengekstrak 48 desa resmi di Kecamatan Kintamani!


,nama_desa,kecamatan,kunci_kabupaten
0,Desa Abang Songan,Kintamani,KABUPATEN BANGLI
1,Desa Abuan,Kintamani,KABUPATEN BANGLI
2,Desa Awan,Kintamani,KABUPATEN BANGLI
3,Desa Bantang,Kintamani,KABUPATEN BANGLI
4,Desa Banua,Kintamani,KABUPATEN BANGLI
5,Desa Batu Dinding,Kintamani,KABUPATEN BANGLI
6,Desa Batukaang,Kintamani,KABUPATEN BANGLI
7,Desa Batur Selatan,Kintamani,KABUPATEN BANGLI
8,Desa Batur Tengah,Kintamani,KABUPATEN BANGLI
9,Desa Batur Utara,Kintamani,KABUPATEN BANGLI


In [26]:
# ==============================================================================
# SEL 5: SOAL 5 - PENGGABUNGAN DATA TERVALIDASI (1:1 dan 1:M) & METADATA
# ==============================================================================

# 1. Filter entitas Kabupaten Bangli dari dataset terbuka (Tepat 1 baris)
df_bangli = df_wilayah_raw[
    df_wilayah_raw["nama_kabupaten"].str.strip().str.upper() == "KABUPATEN BANGLI"
].copy()
df_bangli["kunci_kabupaten"] = "KABUPATEN BANGLI"
assert len(df_bangli) == 1, "Data master Kabupaten Bangli harus tepat 1 baris"

# 2. Ringkasan Cuaca API (1 baris agregasi cuaca Kintamani)
with open(path_api_raw, "r", encoding="utf-8") as f:
    api_json = json.load(f)

df_cuaca = pd.DataFrame(api_json["daily"])
df_ringkasan_cuaca = pd.DataFrame([{
    "kunci_kabupaten": "KABUPATEN BANGLI",
    "periode_observasi_cuaca": f"{df_cuaca['time'].min()} s.d. {df_cuaca['time'].max()}",
    "rata_suhu_maks_c": round(df_cuaca["temperature_2m_max"].mean(), 2),
    "rata_suhu_min_c": round(df_cuaca["temperature_2m_min"].mean(), 2),
    "total_curah_hujan_mm": round(df_cuaca["precipitation_sum"].sum(), 2)
}])

# TAHAP MERGE 1: Wilayah (1 baris) + Cuaca (1 baris) -> VALIDASI one_to_one
df_wilayah_cuaca = pd.merge(
    df_bangli,
    df_ringkasan_cuaca,
    on="kunci_kabupaten",
    how="inner",
    validate="one_to_one"
)

# TAHAP MERGE 2: Gabung dengan 48 Desa Kintamani -> VALIDASI one_to_many
# Kunci: 1 Kabupaten Bangli memuat BANYAK (many) Desa di Kintamani
df_final = pd.merge(
    df_wilayah_cuaca,
    df_desa,
    on="kunci_kabupaten",
    how="inner",
    validate="one_to_many"
)

# 3. Verifikasi Tidak Ada Duplikasi
duplikasi = df_final.duplicated().sum()
print(f"Pengecekan Duplikasi: Ditemukan {duplikasi} baris duplikat.")
assert duplikasi == 0, "Ditemukan duplikasi data!"

# 4. Simpan Hasil Akhir
path_output_final = "data/hasil_penggabungan_kintamani.csv"
df_final.to_csv(path_output_final, index=False)

print("\n" + "="*60)
print(f"HASIL PENGGABUNGAN TERVALIDASI: {len(df_final)} BARIS x {df_final.shape[1]} KOLOM")
print("="*60)
display(df_final.head(10))

# 5. Metadata Hasil Penggabungan
print("\n--- METADATA DATASET HASIL PENGGABUNGAN ---")
for col in df_final.columns:
    print(f"- Kolom '{col}': Tipe = {df_final[col].dtype}, Contoh = {df_final[col].iloc[0]}")

Pengecekan Duplikasi: Ditemukan 0 baris duplikat.

HASIL PENGGABUNGAN TERVALIDASI: 48 BARIS x 10 KOLOM


,kode_kabupaten,kode_provinsi,nama_kabupaten,kunci_kabupaten,periode_observasi_cuaca,rata_suhu_maks_c,rata_suhu_min_c,total_curah_hujan_mm,nama_desa,kecamatan
0,5106,51,KABUPATEN BANGLI,KABUPATEN BANGLI,2024-01-01 s.d. 2024-01-14,24.4,17.59,41.9,Desa Abang Songan,Kintamani
1,5106,51,KABUPATEN BANGLI,KABUPATEN BANGLI,2024-01-01 s.d. 2024-01-14,24.4,17.59,41.9,Desa Abuan,Kintamani
2,5106,51,KABUPATEN BANGLI,KABUPATEN BANGLI,2024-01-01 s.d. 2024-01-14,24.4,17.59,41.9,Desa Awan,Kintamani
3,5106,51,KABUPATEN BANGLI,KABUPATEN BANGLI,2024-01-01 s.d. 2024-01-14,24.4,17.59,41.9,Desa Bantang,Kintamani
4,5106,51,KABUPATEN BANGLI,KABUPATEN BANGLI,2024-01-01 s.d. 2024-01-14,24.4,17.59,41.9,Desa Banua,Kintamani
5,5106,51,KABUPATEN BANGLI,KABUPATEN BANGLI,2024-01-01 s.d. 2024-01-14,24.4,17.59,41.9,Desa Batu Dinding,Kintamani
6,5106,51,KABUPATEN BANGLI,KABUPATEN BANGLI,2024-01-01 s.d. 2024-01-14,24.4,17.59,41.9,Desa Batukaang,Kintamani
7,5106,51,KABUPATEN BANGLI,KABUPATEN BANGLI,2024-01-01 s.d. 2024-01-14,24.4,17.59,41.9,Desa Batur Selatan,Kintamani
8,5106,51,KABUPATEN BANGLI,KABUPATEN BANGLI,2024-01-01 s.d. 2024-01-14,24.4,17.59,41.9,Desa Batur Tengah,Kintamani
9,5106,51,KABUPATEN BANGLI,KABUPATEN BANGLI,2024-01-01 s.d. 2024-01-14,24.4,17.59,41.9,Desa Batur Utara,Kintamani



--- METADATA DATASET HASIL PENGGABUNGAN ---
- Kolom 'kode_kabupaten': Tipe = int64, Contoh = 5106
- Kolom 'kode_provinsi': Tipe = int64, Contoh = 51
- Kolom 'nama_kabupaten': Tipe = object, Contoh = KABUPATEN BANGLI
- Kolom 'kunci_kabupaten': Tipe = object, Contoh = KABUPATEN BANGLI
- Kolom 'periode_observasi_cuaca': Tipe = object, Contoh = 2024-01-01 s.d. 2024-01-14
- Kolom 'rata_suhu_maks_c': Tipe = float64, Contoh = 24.4
- Kolom 'rata_suhu_min_c': Tipe = float64, Contoh = 17.59
- Kolom 'total_curah_hujan_mm': Tipe = float64, Contoh = 41.9
- Kolom 'nama_desa': Tipe = object, Contoh = Desa Abang Songan
- Kolom 'kecamatan': Tipe = object, Contoh = Kintamani


## 🛡️ Data Ethics Checklist (Evaluasi Etika & Tata Kelola Data)

### 1. Kepatuhan Sumber & Lisensi Data (*Source & License Compliance*)
- [x] **Larangan Platform Komersial/Terbatas:** Tidak mengambil data dari situs akomodasi/booking komersial yang dilarang (seperti Airbnb, Agoda, Booking.com, Traveloka, dll.).
- [x] **Dataset Terbuka Riil (>100 Baris):** Menggunakan berkas dataset terbuka asli wilayah administratif Indonesia (`dataset_wilayah_indonesia_raw.csv`, >500 baris) dari repositori terbuka dengan lisensi *Public Domain / Open Data*.
- [x] **API Publik Resmi:** Menggunakan layanan Open-Meteo Historical Weather API di bawah lisensi *Creative Commons Attribution 4.0 (CC-BY 4.0)*.
- [x] **Web Scraping Terbuka:** Mengekstraksi tabel daftar desa resmi Kecamatan Kintamani dari Wikipedia Indonesia di bawah lisensi *CC BY-SA 4.0*.

### 2. Etika Akses Web (*Web Access & Crawling Etiquette*)
- [x] **Pemeriksaan `robots.txt`:** Melakukan pengecekan kebijakan `robots.txt` setiap domain sumber sebelum mengirimkan permintaan HTTP.
- [x] **Transparansi Identitas (`User-Agent`):** Menyertakan header `User-Agent` berisi nama proyek akademik (`INSTIKI-DataAcquisition-UTS/1.0`), tautan institusi (`https://instiki.ac.id`), dan email resmi kampus mahasiswa (`2501010046@std.instiki.ac.id`).
- [x] **Pembatasan Laju Akses (*Rate Limiting*):** Menerapkan jeda waktu sopan (`time.sleep(2)` hingga `time.sleep(3)`) di antara setiap permintaan.
- [x] **Manajemen Koneksi (`Timeout` & `Retry`):** Menggunakan `timeout=15` detik serta mekanisme *exponential backoff retry*.

### 3. Privasi & Perlindungan Data Pribadi (*Privacy & Data Protection*)
- [x] **Bebas Informasi Identitas Pribadi (Non-PII):** Seluruh dataset hanya memuat kode/nama wilayah administratif, daftar nama desa, dan parameter cuaca historis; tidak ada data pribadi individu yang dikumpulkan.

### 4. Integritas, Provenance & Penyimpanan Data (*Data Integrity, Provenance & Storage*)
- [x] **Pemisahan Data Mentah (*Immutability*):** Seluruh berkas mentah disimpan utuh tanpa perubahan di direktori `data/raw/`.
- [x] **Penggabungan Tervalidasi (*Schema & Join Validation*):** Proses penggabungan data (`pd.merge`) menggunakan kunci relasi `KABUPATEN BANGLI` dengan validasi ketat (`validate='one_to_one'` dan `validate='one_to_many'`).
- [x] **Pencatatan Provenance & Verifikasi Hash:** Setiap berkas dilengkapi metadata URL sumber, waktu akses, lisensi, perhitungan ukuran penyimpanan (*storage estimation*), dan nilai *checksum* **MD5** yang dihasilkan langsung oleh kode.